# 01 · Preparar datos — ISIC 2019 → arreglos 224×224

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Salvadrn/isic-pi/blob/main/notebooks/01_preparar_datos.ipynb)

Sesión 1 del proyecto **isic-pi**: un clasificador de lesiones de piel que se entrena en Colab y corre en una Raspberry Pi 5.

**Qué hace este notebook**
1. Descarga `Atoany/ISIC2019` de Hugging Face (≈ 9.8 GB, solo la primera vez).
2. Reduce cada imagen a 224×224 (recorte central cuadrado + resize bilineal). Es la **misma función que usará la Raspberry**, así el modelo ve lo mismo al entrenar y al clasificar.
3. Aparta una parte de validación, estratificada por clase.
4. Guarda todo en tu Google Drive (`Mi unidad/isic-pi/datos`, ≈ 4 GB) para que el notebook 02 lo use sin repetir nada.

**Entorno de ejecución:** CPU basta, no gastes GPU aquí.

> Primera vez: pon `MODO_PRUEBA = True` en la celda de configuración. Usa ~30 imágenes por clase y guarda en otra carpeta; sirve para comprobar que todo corre antes de esperar la corrida completa.

## 1. Configuración

In [ ]:
MODO_PRUEBA = False        # True: ~30 imágenes por clase, guarda en otra carpeta (solo para comprobar que corre)
FORZAR = False             # True: rehacer aunque ya existan datos preparados en Drive
USAR_DRIVE = True          # True: guarda en Mi unidad/CARPETA_DRIVE (persiste entre sesiones)
CARPETA_DRIVE = "isic-pi"

DATASET = "Atoany/ISIC2019"
TAMANO = 224               # lado de entrada del modelo (MobileNetV2)
FRAC_VALIDACION = 0.10     # parte de train que se aparta para validar
SEMILLA = 42

## 2. Instalar y montar Drive

In [ ]:
%pip install -q -U datasets

In [ ]:
import json, os, shutil, sys, time
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / CARPETA_DRIVE
elif EN_COLAB:
    BASE = Path("/content") / CARPETA_DRIVE
else:
    BASE = Path.cwd() / CARPETA_DRIVE              # fuera de Colab (pruebas locales)

SALIDA = BASE / ("datos_prueba" if MODO_PRUEBA else "datos")      # destino final, persistente
TRABAJO = (Path("/content") if EN_COLAB else Path.cwd()) / "trabajo" / SALIDA.name   # disco local, rápido
SALIDA.mkdir(parents=True, exist_ok=True)
TRABAJO.mkdir(parents=True, exist_ok=True)

if (SALIDA / "meta.json").exists() and not FORZAR:
    raise RuntimeError(f"Ya hay datos preparados en {SALIDA}. Pasa al notebook 02, o pon FORZAR = True para rehacerlos.")
print("Destino final:", SALIDA)
print("Carpeta de trabajo:", TRABAJO)

## 3. Cargar el dataset

Columnas: `image` (PIL) y `label` (texto). Los splits `train` y `test` vienen ya hechos en Hugging Face.

In [ ]:
import numpy as np
from datasets import load_dataset

dataset = load_dataset(DATASET)
train_raw, test_raw = dataset["train"], dataset["test"]
print(train_raw.features)
print(f"Train: {len(train_raw):,} | Test: {len(test_raw):,}")


def submuestra(ds, por_clase):
    etiquetas = np.array(list(ds["label"]))
    rng = np.random.default_rng(SEMILLA)
    elegidos = []
    for c in sorted(set(etiquetas)):
        idx = np.flatnonzero(etiquetas == c)
        elegidos += [int(i) for i in rng.choice(idx, size=min(por_clase, len(idx)), replace=False)]
    return ds.select(sorted(elegidos))


if MODO_PRUEBA:
    train_raw = submuestra(train_raw, 30)
    test_raw = submuestra(test_raw, 10)
    print(f"MODO_PRUEBA → Train: {len(train_raw)} | Test: {len(test_raw)}")

## 4. Clases y distribución

El índice de cada clase es su posición en la lista alfabética `CLASES`. Ese orden se guarda en `clases.txt` y es el que usará la Raspberry.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

NOMBRES = {
    "AK": "Queratosis actínica", "BCC": "Carcinoma basocelular", "BKL": "Queratosis benigna",
    "DF": "Dermatofibroma", "MEL": "Melanoma", "NV": "Nevo melanocítico",
    "SCC": "Carcinoma escamoso", "VASC": "Lesión vascular",
}

etiquetas_train = list(train_raw["label"])
etiquetas_test = list(test_raw["label"])
CLASES = sorted(set(etiquetas_train))
K = len(CLASES)
IDX_CLASE = {c: i for i, c in enumerate(CLASES)}

solo_test = sorted(set(etiquetas_test) - set(CLASES))
if solo_test:
    print("⚠ Etiquetas que solo aparecen en test (esas imágenes se descartan):", solo_test)
    ok = [e in IDX_CLASE for e in etiquetas_test]
    test_raw = test_raw.select([i for i, v in enumerate(ok) if v])
    etiquetas_test = [e for e, v in zip(etiquetas_test, ok) if v]

conteo = pd.DataFrame({
    "train": pd.Series(etiquetas_train).value_counts(),
    "test": pd.Series(etiquetas_test).value_counts(),
}).reindex(CLASES).fillna(0).astype(int)
conteo["% train"] = (100 * conteo["train"] / conteo["train"].sum()).round(1)
conteo["descripción"] = [NOMBRES.get(c, c) for c in CLASES]
display(conteo)

conteo[["train", "test"]].plot(kind="bar", figsize=(9, 4), title="Imágenes por clase")
plt.ylabel("Número de imágenes")
plt.xticks(rotation=0)
plt.show()

## 5. Reducir a 224×224 y guardar como arreglos

`preparar_imagen` es **la función compartida con la Raspberry** (`pi/clasificar.py` tiene una copia idéntica). Si algún día cambias algo aquí, cámbialo allá también y vuelve a entrenar.

Se guarda como `uint8` (0–255) y el modelo hará su propio escalado interno, así la Raspberry solo tiene que mandarle los píxeles tal cual.

In [ ]:
import io
from concurrent.futures import ThreadPoolExecutor
from datasets import Image as HFImage
from PIL import Image, ImageOps
from tqdm.auto import tqdm


def preparar_imagen(img, tamano=TAMANO):
    """EXIF → RGB → recorte central cuadrado → resize bilineal → uint8 (tamano, tamano, 3).
    Idéntica a la de pi/clasificar.py."""
    img = ImageOps.exif_transpose(img).convert("RGB")
    img = ImageOps.fit(img, (tamano, tamano), method=Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)


def _desde_bytes(d):
    if d["bytes"] is not None:
        return preparar_imagen(Image.open(io.BytesIO(d["bytes"])))
    return preparar_imagen(Image.open(d["path"]))


def convertir(split, nombre):
    n = len(split)
    ruta = TRABAJO / f"{nombre}_x.npy"
    X = np.lib.format.open_memmap(ruta, mode="w+", dtype=np.uint8, shape=(n, TAMANO, TAMANO, 3))
    crudo = split.cast_column("image", HFImage(decode=False))   # bytes sin decodificar: se decodifica en hilos
    pos = 0
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=os.cpu_count() or 2) as pool:
        for lote in tqdm(crudo.iter(batch_size=256), total=-(-n // 256), desc=nombre):
            arr = list(pool.map(_desde_bytes, lote["image"]))
            X[pos:pos + len(arr)] = np.stack(arr)
            pos += len(arr)
    X.flush()
    assert pos == n, f"{nombre}: se convirtieron {pos} de {n}"
    print(f"{nombre}: {n:,} imágenes en {time.time() - t0:.0f} s → {ruta.name} ({ruta.stat().st_size / 1e9:.2f} GB)")
    return ruta


y_train = np.array([IDX_CLASE[e] for e in etiquetas_train], dtype=np.int16)
y_test = np.array([IDX_CLASE[e] for e in etiquetas_test], dtype=np.int16)

necesario = (len(y_train) + len(y_test)) * TAMANO * TAMANO * 3
try:
    libre = shutil.disk_usage(SALIDA).free
    print(f"Espacio que ocupará: {necesario / 1e9:.2f} GB | libre reportado en destino: {libre / 1e9:.1f} GB")
    if libre < necesario * 1.05:
        print("⚠ Parece que no cabe en el destino. Libera espacio en Drive o pon USAR_DRIVE = False.")
except OSError:
    print(f"Espacio que ocupará: {necesario / 1e9:.2f} GB (no pude consultar el espacio libre del destino)")

convertir(train_raw, "train")
convertir(test_raw, "test")

## 6. Validación estratificada

Se aparta `FRAC_VALIDACION` de train, manteniendo la proporción de cada clase. **Test no se toca** hasta la evaluación final.

> Ojo: la partición train/test ya venía hecha en el dataset y no pude verificar cómo se hizo. Si fotos de la misma lesión quedaron repartidas entre train y test, la métrica final saldrá más optimista de lo real.

In [ ]:
from sklearn.model_selection import train_test_split

idx_tr, idx_val = train_test_split(
    np.arange(len(y_train)), test_size=FRAC_VALIDACION, stratify=y_train, random_state=SEMILLA
)
idx_tr, idx_val = np.sort(idx_tr), np.sort(idx_val)

reparto = pd.DataFrame({
    "train": np.bincount(y_train[idx_tr], minlength=K),
    "val": np.bincount(y_train[idx_val], minlength=K),
    "test": np.bincount(y_test, minlength=K),
}, index=CLASES)
display(reparto)
print(f"train {len(idx_tr):,} | val {len(idx_val):,} | test {len(y_test):,}")

## 7. Mirar lo que verá el modelo

Cinco imágenes por clase, **ya reducidas** (leídas del arreglo, no del dataset original). Si algo se ve mal (colores raros, recortes absurdos), mejor enterarse ahora.

In [ ]:
X_train = np.load(TRABAJO / "train_x.npy", mmap_mode="r")
rng = np.random.default_rng(SEMILLA)
fig, ejes = plt.subplots(K, 5, figsize=(10, 2 * K), squeeze=False)
for fila, c in enumerate(CLASES):
    idx = np.flatnonzero(y_train == fila)
    for col in range(5):
        ejes[fila, col].axis("off")
    for col, i in enumerate(rng.choice(idx, size=min(5, len(idx)), replace=False)):
        ejes[fila, col].imshow(X_train[i])
    ejes[fila, 0].set_title(c, loc="left", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Guardar en el destino

In [ ]:
import PIL

np.save(TRABAJO / "train_y.npy", y_train)
np.save(TRABAJO / "test_y.npy", y_test)
np.savez(TRABAJO / "split.npz", idx_tr=idx_tr, idx_val=idx_val)
(TRABAJO / "clases.txt").write_text("\n".join(CLASES) + "\n")

meta = {
    "dataset": DATASET,
    "modo_prueba": MODO_PRUEBA,
    "tamano": TAMANO,
    "clases": CLASES,
    "n_train": int(len(y_train)),
    "n_val": int(len(idx_val)),
    "n_test": int(len(y_test)),
    "frac_validacion": FRAC_VALIDACION,
    "semilla": SEMILLA,
    "preprocesado": "exif_transpose, RGB, ImageOps.fit(BILINEAR, centrado), uint8",
    "pillow": PIL.__version__,
    "numpy": np.__version__,
    "fecha": time.strftime("%Y-%m-%d %H:%M"),
}
(TRABAJO / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2))

if SALIDA.resolve() != TRABAJO.resolve():
    (SALIDA / "meta.json").unlink(missing_ok=True)       # meta.json se copia al final: marca "terminado"
    for nombre in ["train_x.npy", "train_y.npy", "test_x.npy", "test_y.npy", "split.npz", "clases.txt", "meta.json"]:
        shutil.copy2(TRABAJO / nombre, SALIDA / nombre)

for p in sorted(SALIDA.iterdir()):
    print(f"{p.name:14s} {p.stat().st_size / 1e6:9.1f} MB")

# Comprobación: lo guardado se lee igual
assert np.load(SALIDA / "train_x.npy", mmap_mode="r").shape == (len(y_train), TAMANO, TAMANO, 3)
assert np.load(SALIDA / "test_x.npy", mmap_mode="r").shape == (len(y_test), TAMANO, TAMANO, 3)
print("\n✓ Listo. Siguiente: notebook 02 (entrenar).")

## Qué quedó en Drive

```
Mi unidad/isic-pi/datos/
├── train_x.npy   (N, 224, 224, 3) uint8
├── train_y.npy   (N,) índice de clase
├── test_x.npy
├── test_y.npy
├── split.npz     idx_tr, idx_val (posiciones dentro de train)
├── clases.txt    una clase por línea; el índice es la posición
└── meta.json     parámetros y versiones (se escribe al final: si existe, todo terminó bien)
```

Siguiente sesión: **02 · Entrenar** (MobileNetV2 con transfer learning y pesos por clase).